In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd

from shapely.geometry import LineString, MultiLineString
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from shapely.geometry import box
import matplotlib.colors as mcolors

from dask import delayed, compute
from tqdm import tqdm
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar

# ============================
# User settings
# ============================

# Generator site list
gen_csv = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv")

boco_ds = "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/C2_boco_wind_vec.nc"
clust_ds = "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/C2_cluster_wind_vec.nc"

In [2]:
client = Client(n_workers=12,
    threads_per_worker=1,
    memory_limit=f"{int(5)}GB"
)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 12
Total threads: 12,Total memory: 55.88 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:36329,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34483,Total threads: 1
Dashboard: /proxy/41615/status,Memory: 4.66 GiB
Nanny: tcp://127.0.0.1:40333,


2025-09-03 14:02:20,044 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 210e3aee7390ea926d5c75dd0ab1d3b7 initialized by task ('open_dataset-rechunk-transfer-7c0e4342aad0f3bd4f20af6d1cba6a0c', 0, 0, 0, 99, 0, 0) executed on worker tcp://127.0.0.1:39609
2025-09-03 14:02:20,054 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 64ee0ca0e28237836ec00ef5697cfda8 initialized by task ('open_dataset-rechunk-transfer-3e80611619b792e2f7132fda193e4d80', 0, 0, 0, 34, 0, 0) executed on worker tcp://127.0.0.1:34011
2025-09-03 14:03:39,594 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 210e3aee7390ea926d5c75dd0ab1d3b7 deactivated due to stimulus 'task-finished-1756872219.5927975'
2025-09-03 14:03:50,988 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 64ee0ca0e28237836ec00ef5697cfda8 deactivated due to stimulus 'task-finished-1756872230.9863818'


In [3]:
# Statistically significant w ssmin=20, and Mann-Whitney 
cluster = ['TARALGA1','CROOKWF2',
            'GULLRWF1',
            'GUNNING1',
            'BANGOWF1',
            'WOODLWN1',
            'BOCORWF1']

cluster = gen_csv[gen_csv['DUID'].isin(cluster)][['DUID','lat','lon']]

In [4]:
extent = [147.5, 151, -38.5, -33.5]
lon_min, lon_max, lat_min, lat_max = extent

In [5]:
ds = xr.open_mfdataset(boco_ds, chunks='auto', engine='h5netcdf', parallel=True)
ds = ds.chunk({'time': -1})

# Convert to Australia/Sydney
local_time = (
    pd.DatetimeIndex(ds.time.values)
    .tz_localize("UTC")
    .tz_convert("Australia/Sydney")
)

# Drop tzinfo so xarray can store it
local_time_naive = local_time.tz_localize(None)

# Assign back to dataset
ds = ds.assign_coords(time=local_time_naive)
ds

<xarray.Dataset> Size: 60GB
Dimensions:  (time: 2832, lat: 1018, lon: 1298)
Coordinates:
    height   float64 8B ...
  * lon      (lon) float64 10kB 108.0 108.1 108.1 108.1 ... 159.8 159.9 159.9
  * lat      (lat) float64 8kB -45.69 -45.65 -45.61 -45.57 ... -5.09 -5.05 -5.01
    crs      int32 4B ...
  * time     (time) datetime64[ns] 23kB 2015-12-19T11:00:00 ... 2023-12-12T10...
Data variables:
    ua100m   (time, lat, lon) float64 30GB dask.array<chunksize=(2832, 100, 100), meta=np.ndarray>
    va100m   (time, lat, lon) float64 30GB dask.array<chunksize=(2832, 100, 100), meta=np.ndarray>
Attributes: (12/60)
    axiom_version:             0.1.0
    axiom_schemas_version:     0.1.0
    axiom_schema:              cordex-1H.json
    productive_version:        462b27f
    variable_version:          v20240809
    Conventions:               CF-1.10, ACDD-1.3
    ...                        ...
    date_modified:             2024-10-11T00:33:24Z
    date_metadata_modified:    2024-10-11T00:33:24Z
    history:                   Tue Aug 06 06:47:11 2024: /g/data/access/ngm/m...
    references:                https://doi.org/10.25914/1x6g-2v48
    license:                   https://doi.org/10.25914/1x6g-2v48
    acknowledgement:           The production of BARRA2 was supported with fu...

In [6]:
# Crop to extent
ds_subset = ds.sel(**{
    'lon': slice(lon_min, lon_max),
    'lat': slice(lat_min, lat_max)
})

ds_subset

# Lazy hourly mean
hourly_composite =  ds_subset.groupby("time.hour").mean()

# Compute in parallel
with ProgressBar():
    hourly_composite = hourly_composite.compute()
    hourly_composite = hourly_composite.assign_coords(hour=("hour", np.arange(24)))

Functions which calculate quantities from vector fields

In [7]:
def calc_divergence(ds):
    """
    Compute divergence of u/v on a spherical Earth using xarray.
    ds should be a Dataset with dimensions lat x lon (or time x lat x lon).
    """
    R = 6371000  # Earth radius in meters
    
    # lat_rad shape (lat,)
    lat_rad = np.deg2rad(ds['lat'].values)
    
    dx_1d = np.gradient(ds['lon'].values) * (np.pi/180) * R
    dx2d = dx_1d[None, :] * np.cos(lat_rad[:, None])  # shape (lat, lon)
    
    dy2d = np.gradient(ds['lat'].values) * (np.pi/180) * R
    dy2d = dy2d[:, None]  # shape (lat, 1) to broadcast with u/v

    # Compute divergence
    divergence = (ds['ua100m'].differentiate('lon') / dx2d +
                  ds['va100m'].differentiate('lat') / dy2d)
    
    return divergence

In [8]:
def okubo_weiss(ds, u_name="ua100m", v_name="va100m", lat_name="lat", lon_name="lon"):
    R = 6371000.0
    lat_vals = ds[lat_name].values
    lon_vals = ds[lon_name].values
    lat_rad = np.deg2rad(lat_vals)

    dlat = np.gradient(lat_vals) * np.pi/180 * R
    dlon = np.gradient(lon_vals) * np.pi/180 * R

    # dx/dy 2D arrays matching full grid
    dx2d = xr.DataArray(dlon[None, :] * np.cos(lat_rad[:, None]),
                        dims=[lat_name, lon_name],
                        coords={lat_name: lat_vals, lon_name: lon_vals})
    dy2d = xr.DataArray(dlat[:, None] * np.ones(len(lon_vals)),
                        dims=[lat_name, lon_name],
                        coords={lat_name: lat_vals, lon_name: lon_vals})

    du_dlon = ds[u_name].differentiate(lon_name) * np.pi/180
    du_dlat = ds[u_name].differentiate(lat_name) * np.pi/180
    dv_dlon = ds[v_name].differentiate(lon_name) * np.pi/180
    dv_dlat = ds[v_name].differentiate(lat_name) * np.pi/180

    dudx = du_dlon / dx2d
    dudy = du_dlat / dy2d
    dvdx = dv_dlon / dx2d
    dvdy = dv_dlat / dy2d

    s_n = dudx - dvdy
    s_s = dudy + dvdx
    omega = dvdx - dudy
    OW = s_n**2 + s_s**2 - omega**2

    return xr.Dataset({"s_n": s_n, "s_s": s_s, "omega": omega, "OW": OW})


In [9]:
def calc_curl(hourly_composite):
    R = 6371000  # Earth radius in meters
        
    # lat_rad shape (lat,)
    lat_rad = np.deg2rad(hourly_composite['lat'].values)
    
    # dx along longitude, in meters
    dx_1d = np.gradient(hourly_composite['lon'].values) * (np.pi/180) * R
    dx2d = dx_1d[None, :] * np.cos(lat_rad[:, None])  # shape (lat, lon)
    
    # dy along latitude, in meters
    dy2d = np.gradient(hourly_composite['lat'].values) * (np.pi/180) * R
    dy2d = dy2d[:, None]  # shape (lat, 1) to broadcast
    
    # Compute curl (z-component)
    curl_z = (hourly_composite['va100m'].differentiate('lon') / dx2d -
              hourly_composite['ua100m'].differentiate('lat') / dy2d)
    return curl_z

Plotting funcitons for vector field, divergence, Okubo-Weiss, and curl

In [10]:
def plot_vector_frame(u, v, lat, lon, t, output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_hourly_composites/',
               quiver_scale=None, extent=extent, cluster=cluster, highlight_id='BOCORWF1'):
    """
    Plot wind vectors with optional cluster points.
    
    cluster: DataFrame with columns ['ID', 'lat', 'lon']
    highlight_id: specific ID to highlight in red
    """
    speed = np.sqrt(u**2 + v**2)
    plt.figure(figsize=(10,8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, lw=1.5)
    ax.add_feature(cfeature.BORDERS, linestyle='-', lw=1.5)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.5)
    
    # Contour of wind speed
    plt.contourf(lon, lat, np.sqrt(u**2 + v**2),  # use full arrays here
                 cmap='cividis', transform=ccrs.PlateCarree())
    plt.colorbar(label='Wind speed (m/s)')
    
    # Quiver vectors
    plt.quiver(lon[::step], lat[::step], u[::step, ::step], v[::step, ::step],
               scale=quiver_scale, color='white', transform=ccrs.PlateCarree())
    
    # Plot cluster points
    if cluster is not None:
        # All points in orange
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", alpha=0.6, s=50, label="Wind Farms", zorder=5, transform=ccrs.PlateCarree())
        
        # Highlight one point in red
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'], 
                       color="red", alpha=0.6, s=80, label=f"ID {highlight_id}", zorder=6,
                       transform=ccrs.PlateCarree())

    plt.title(f'Composite of heatwave-day wind vectors at hour {str(t)} (AEST)')
    
    filename = make_filename(t, output_dir)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {filename}')
    return filename


def make_filename(t, output_dir, prefix="wind"):
    try:
        # If t is datetime-like, use date formatting
        dt_str = np.datetime_as_string(t, unit='m')
        dt_str = dt_str.replace('-', '')[2:8] + '_' + dt_str[11:13] + dt_str[14:16]
    except Exception:
        # If it's just an index/hour, format as hour
        if isinstance(t, (int, np.integer)):
            dt_str = f"hour{t:02d}"
        else:
            dt_str = str(t).replace(":", "").replace(" ", "_")
    
    return os.path.join(output_dir, f"{prefix}_{dt_str}.png")


In [11]:
def plot_divergence_frame(divergence, lat, lon, t, cluster=cluster, highlight_id='BOCORWF1',
                          shapefile_gdf=None,
                          output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/divergence_composite',
                          extent=extent):
    """
    Plot scalar divergence with optional cluster points and contour shapefile.
    """
    fig, ax = plt.subplots(figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    
    # Base map
    ax.add_feature(cfeature.COASTLINE, lw=1.0)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.0)
    
    # Divergence field
    # Symmetric normalization around zero
    norm = mcolors.TwoSlopeNorm(vmin=np.nanmin(divergence),
                                vcenter=0,
                                vmax=np.nanmax(divergence))
    
    cf = ax.contourf(lon, lat, divergence, cmap='coolwarm', levels=21, norm=norm,
                     transform=ccrs.PlateCarree(), zorder=1)
    plt.colorbar(cf, ax=ax, label='Divergence (1/s)')
    
    # --- Contours (from shapefile) ---
    if shapefile_gdf is not None and not shapefile_gdf.empty:
        for geom in shapefile_gdf.geometry:
            ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                              facecolor='none', edgecolor='black',
                              linewidth=0.5, zorder=3)
    
    # Cluster points
    if cluster is not None:
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", edgecolor='black', s=50,
                   label="Wind Farms", zorder=4, transform=ccrs.PlateCarree())
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'],
                       color="red", edgecolor='black', s=60,
                       label=f"ID {highlight_id}", zorder=5, transform=ccrs.PlateCarree())
    
    plt.title(f'Divergence of composite wind field at hour {str(t)} (AEST)')
    
    # Save
    filename = make_filename(t, output_dir)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return filename

In [20]:
def plot_ow_frame(df, lat, lon, t, cluster=cluster, highlight_id='BOCORWF1',
                          shapefile_gdf=None,
                          output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/okubo-weiss',
                          extent=extent):
    """
    Plot scalar OW parameter with optional cluster points and contour shapefile.
    """
    fig, ax = plt.subplots(figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    
    # Base map
    ax.add_feature(cfeature.COASTLINE, lw=1.0)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.0)
    
    norm = mcolors.TwoSlopeNorm(vmin=np.nanmin(df),
                                vcenter=0,
                                vmax=np.nanmax(df))
    
    cf = ax.contourf(lon, lat, df, cmap='seismic', levels=21, norm=norm,
                     transform=ccrs.PlateCarree(), zorder=1)
    plt.colorbar(cf, ax=ax, label='Okubo-Weiss parameter')
    
    # --- Contours (from shapefile) ---
    if shapefile_gdf is not None and not shapefile_gdf.empty:
        for geom in shapefile_gdf.geometry:
            ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                              facecolor='none', edgecolor='black',
                              linewidth=0.5, zorder=3)
    
    # Cluster points
    if cluster is not None:
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", edgecolor='black', s=50,
                   label="Wind Farms", zorder=4, transform=ccrs.PlateCarree())
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'],
                       color="red", edgecolor='black', s=60,
                       label=f"ID {highlight_id}", zorder=5, transform=ccrs.PlateCarree())
    
    plt.title(f'Okubo-Weiss from composite at hour {str(t)} (AEST)')
    
    # Save
    filename = make_filename(t, output_dir)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return filename

In [13]:
def plot_curl_frame(df, lat, lon, t, cluster=cluster, highlight_id='BOCORWF1',
                          shapefile_gdf=None,
                          output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/curl',
                          extent=extent):
    """
    Plot scalar curl parameter with optional cluster points and contour shapefile.
    """
    fig, ax = plt.subplots(figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    
    # Base map
    ax.add_feature(cfeature.COASTLINE, lw=1.0)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.0)
    
    norm = mcolors.TwoSlopeNorm(vmin=np.nanmin(df),
                                vcenter=0,
                                vmax=np.nanmax(df))
    
    cf = ax.contourf(lon, lat, df, cmap='PiYG', levels=21, norm=norm,
                     transform=ccrs.PlateCarree(), zorder=1)
    plt.colorbar(cf, ax=ax, label=' Curl (1/s): Pink=ACW, Green=CW ')
    
    # --- Contours (from shapefile) ---
    if shapefile_gdf is not None and not shapefile_gdf.empty:
        for geom in shapefile_gdf.geometry:
            ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                              facecolor='none', edgecolor='black',
                              linewidth=0.5, zorder=3)
    
    # Cluster points
    if cluster is not None:
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", edgecolor='black', s=50,
                   label="Wind Farms", zorder=4, transform=ccrs.PlateCarree())
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'],
                       color="red", edgecolor='black', s=60,
                       label=f"ID {highlight_id}", zorder=5, transform=ccrs.PlateCarree())
    
    plt.title(f'Curl from composite at hour {str(t)} (AEST)')
    
    # Save
    filename = make_filename(t, output_dir)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return filename

Getting shapefile and subset for plotting

In [21]:
# Load shapefile
gdf = gpd.read_file('/g/data/ng72/ms5578/ID_HW_BARRA/data/raw/contours/aus25cgd_l.shp').to_crs(epsg=4326)
bbox = box(lon_min, lat_min, lon_max, lat_max)
gdf_clip = gdf[gdf.geometry.intersects(bbox)]

In [15]:
lat_subset = hourly_composite['lat'].where(
    (hourly_composite['lat'] >= lat_min) & (hourly_composite['lat'] <= lat_max),
    drop=True
    ).values

lon_subset = hourly_composite['lon'].where(
    (hourly_composite['lon'] >= lon_min) & (hourly_composite['lon'] <= lon_max),
    drop=True
    ).values

Plotting loops

In [16]:
for t in range(len(hourly_composite.hour)):
    step=2
    # --- Full resolution (for contours) ---
    u_full = hourly_composite['ua100m'].isel(hour=t).sel(
        lat=lat_subset, lon=lon_subset
    ).values
    v_full = hourly_composite['va100m'].isel(hour=t).sel(
        lat=lat_subset, lon=lon_subset
    ).values

    target_nx, target_ny = 60, 90  # about this many arrows across lon/lat
    lon_idx = np.linspace(0, len(lon_subset)-1, target_nx, dtype=int)
    lat_idx = np.linspace(0, len(lat_subset)-1, target_ny, dtype=int)
    
    lon_quiv = lon_subset[lon_idx]
    lat_quiv = lat_subset[lat_idx]
    u_quiv = u_full[np.ix_(lat_idx, lon_idx)]
    v_quiv = v_full[np.ix_(lat_idx, lon_idx)]

    # Time

    # Call plotting function with full fields for contours
    # and stepped ones for quivers
    plot_vector_frame(
        u_quiv, v_quiv, lat_quiv, lon_quiv,  # quiver data
        t, 
        quiver_scale=150,
        extent=extent
    )

Saved: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_hourly_composites/wind_hour00.png
Saved: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_hourly_composites/wind_hour01.png
Saved: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_hourly_composites/wind_hour02.png
Saved: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_hourly_composites/wind_hour03.png
Saved: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_hourly_composites/wind_hour04.png
Saved: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_hourly_composites/wind_hour05.png
Saved: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_hourly_composites/wind_hour06.png
Saved: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_hourly_composites/wind_hour07.png
Saved: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_hourly_composites/wind_hour08.png
Saved: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_hourly_composites/wind_hour09.png


In [17]:
div = calc_divergence(hourly_composite)

# Loop for plotting divergence
for t in range(len(div['hour'])):
    div_time = div.isel(hour=t).values
    
    filestring = plot_divergence_frame(div_time, lat_subset, lon_subset, t,
                                 cluster=cluster,
                                 shapefile_gdf=gdf)
    
    print(f'Saved divergence plots: {filestring}')

Saved divergence plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/divergence_composite/wind_hour00.png
Saved divergence plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/divergence_composite/wind_hour01.png
Saved divergence plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/divergence_composite/wind_hour02.png
Saved divergence plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/divergence_composite/wind_hour03.png
Saved divergence plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/divergence_composite/wind_hour04.png
Saved divergence plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/divergence_composite/wind_hour05.png
Saved divergence plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/divergence_composite/wind_hour06.png
Saved divergence plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/divergence_composite/wind_hour07.png
Saved divergence plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/out

In [22]:
OW = okubo_weiss(hourly_composite, u_name="ua100m", v_name="va100m")['OW']

# Loop for plotting OW
for t in range(len(OW['hour'])):
    slice_time = OW.isel(hour=t).values
    
    filestring = plot_ow_frame(slice_time, lat_subset, lon_subset, t,
                                 cluster=cluster,
                                 shapefile_gdf=gdf)
    
    print(f'Saved OW plots: {filestring}')

Saved OW plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/okubo-weiss/wind_hour00.png
Saved OW plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/okubo-weiss/wind_hour01.png
Saved OW plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/okubo-weiss/wind_hour02.png
Saved OW plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/okubo-weiss/wind_hour03.png
Saved OW plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/okubo-weiss/wind_hour04.png
Saved OW plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/okubo-weiss/wind_hour05.png
Saved OW plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/okubo-weiss/wind_hour06.png
Saved OW plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/okubo-weiss/wind_hour07.png
Saved OW plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/okubo-weiss/wind_hour08.png
Saved OW plots: /g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/okubo-weiss/wind_hour09.png


In [19]:
curl_z = calc_curl(hourly_composite)

for t in range(24):
    filestring = plot_curl_frame(curl_z[t,:,:],
                          lat_subset,
                          lon_subset,
                          t,
                          shapefile_gdf=gdf_clip)
    print(filestring)


/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/curl/wind_hour00.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/curl/wind_hour01.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/curl/wind_hour02.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/curl/wind_hour03.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/curl/wind_hour04.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/curl/wind_hour05.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/curl/wind_hour06.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/curl/wind_hour07.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/curl/wind_hour08.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/curl/wind_hour09.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/curl/wind_hour10.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/curl/wind_hour11.png
/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/curl/wind_hour12.png